In [ ]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.6 MB/s eta 0:00:00


In [ ]:
import os
import json
import re
import time
from getpass import getpass

from groq import Groq

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Groq API key loaded successfully.")

Groq API key loaded successfully.


In [ ]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
print("Groq client initialized.")

Groq client initialized.


In [ ]:
MODEL = "llama-3.3-70b-versatile"

print("Model:", MODEL)

Model: llama-3.3-70b-versatile


In [ ]:
USER_QUERY = """
I am a beginner at drawing. How can I improve my sketching?
"""

print("Test Query:")
print(USER_QUERY)

Test Query:

I am a beginner at drawing. How can I improve my sketching?



In [ ]:
def ask_groq(prompt, temperature=0.7):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature
    )

    return response.choices[0].message.content

In [ ]:
def show_result(title, result):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(result)
    print("=" * 80)

In [ ]:
baseline_prompt = f"""
You are a drawing tutor.

Answer the following user question:

{USER_QUERY}
"""

baseline_output = ask_groq(baseline_prompt)

show_result(
    "ORIGINAL SYSTEM - BEFORE OPTIMIZATION",
    baseline_output
)


ORIGINAL SYSTEM - BEFORE OPTIMIZATION
Welcome to the world of drawing. I'm excited to help you improve your sketching skills. As a beginner, it's essential to start with the basics and build your way up. Here are some tips to get you started:

1. **Practice regularly**: Consistency is key when it comes to improving your drawing skills. Set aside a specific time each day or each week to practice drawing, even if it's just for a few minutes.
2. **Start with simple exercises**: Begin with basic exercises like drawing shapes, lines, and values (light and dark). This will help you develop your fine motor skills and get a feel for the medium you're using.
3. **Use reference images**: Gather reference images of the things you want to draw, such as animals, landscapes, or still-life compositions. This will help you observe the proportions, shapes, and details of the subject.
4. **Learn to see and observe**: Drawing is not just about putting lines on paper; it's about observing and interpretin

In [ ]:
optimization_1_prompt = f"""
You are an expert drawing tutor.

Your goal is to teach the student clearly and practically.

Follow these rules:

1. Identify the main drawing topic.
2. Explain the concept in simple beginner-friendly language.
3. Give step-by-step instructions.
4. Include practical exercises.
5. Give useful drawing tips.
6. Mention common mistakes.
7. Use clear headings and bullet points.
8. Avoid unnecessary repetition.
9. Keep the explanation suitable for a beginner.

Student question:

{USER_QUERY}

Provide a structured tutoring response.
"""

optimization_1_output = ask_groq(
    optimization_1_prompt
)

show_result(
    "OPTIMIZATION 1 - AFTER PROMPT OPTIMIZATION",
    optimization_1_output
)


OPTIMIZATION 1 - AFTER PROMPT OPTIMIZATION
**Welcome to Sketching 101**

### Introduction to Sketching

Sketching is a fundamental skill for any artist, and it's great that you're starting your journey. The main drawing topic we'll be covering is the basics of sketching. Don't worry if you're new to drawing; we'll take it one step at a time.

### Understanding the Concept

Sketching is all about capturing the basic shape and form of an object or scene. It's a quick and simple way to get your ideas down on paper. Think of it as a warm-up exercise for your drawing muscles.

### Step-by-Step Instructions

To get started with sketching, follow these simple steps:

1. **Choose your tools**: You'll need a pencil, eraser, and paper. You can start with a simple graphite pencil (HB or 2B) and a sketchbook.
2. **Observe your subject**: Look at the object or scene you want to sketch. Take note of its basic shape, proportions, and any notable features.
3. **Start with simple shapes**: Break down 

In [ ]:
print("\n")
print("=" * 100)
print("OPTIMIZATION 1 - BEFORE VS AFTER")
print("=" * 100)

print("\n--- BEFORE OPTIMIZATION ---")
print(baseline_output)

print("\n--- AFTER OPTIMIZATION 1 ---")
print(optimization_1_output)

print("\n--- DIFFERENCE ---")

print("""
BEFORE:
- Generic drawing tutor instruction
- No strict response structure
- May give general advice
- No required sections

AFTER:
- Beginner-focused explanation
- Clear topic identification
- Step-by-step instructions
- Practical exercises
- Drawing tips
- Common mistakes
- Structured formatting
""")



OPTIMIZATION 1 - BEFORE VS AFTER

--- BEFORE OPTIMIZATION ---
Welcome to the world of drawing. I'm excited to help you improve your sketching skills. As a beginner, it's essential to start with the basics and build your way up. Here are some tips to get you started:

1. **Practice regularly**: Consistency is key when it comes to improving your drawing skills. Set aside a specific time each day or each week to practice drawing, even if it's just for a few minutes.
2. **Start with simple exercises**: Begin with basic exercises like drawing shapes, lines, and values (light and dark). This will help you develop your fine motor skills and get a feel for the medium you're using.
3. **Use reference images**: Gather reference images of the things you want to draw, such as animals, landscapes, or still-life compositions. This will help you observe the proportions, shapes, and details of the subject.
4. **Learn to see and observe**: Drawing is not just about putting lines on paper; it's about 

In [ ]:
intent_prompt = f"""
You are an intent classification agent for a drawing tutor.

Identify the student's:

- skill
- intent
- level
- topic

Choose the intent from:

definition
improvement
technique
example
practice
materials
comparison
general

Return ONLY valid JSON.

Example:

{{
    "skill": "drawing",
    "intent": "improvement",
    "level": "beginner",
    "topic": "sketching"
}}

Student question:

{USER_QUERY}
"""

intent_result = ask_groq(
    intent_prompt,
    temperature=0
)

intent_result = intent_result.replace(
    "```json", ""
).replace(
    "```", ""
).strip()

print("=" * 80)
print("OPTIMIZATION 2 — INTENT CLASSIFICATION")
print("=" * 80)

print("\nCLASSIFICATION:")
print(intent_result)

OPTIMIZATION 2 — INTENT CLASSIFICATION

CLASSIFICATION:
{
    "skill": "drawing",
    "intent": "improvement",
    "level": "beginner",
    "topic": "sketching"
}


In [ ]:
try:
    classification = json.loads(intent_result)

except:
    classification = {
        "skill": "drawing",
        "intent": "improvement",
        "level": "beginner",
        "topic": "sketching"
    }


intent_tutor_prompt = f"""
You are an expert drawing tutor.

Student information:

Skill: {classification["skill"]}
Intent: {classification["intent"]}
Level: {classification["level"]}
Topic: {classification["topic"]}

Use this information to give a targeted tutoring response.

Student question:

{USER_QUERY}
"""

optimization_2_output = ask_groq(
    intent_tutor_prompt
)

In [ ]:
print("=" * 100)
print("OPTIMIZATION 2 — BEFORE VS AFTER")
print("=" * 100)

print("\nBEFORE:")
print(baseline_output)

print("\nAFTER:")
print(optimization_2_output)

print("\nINTENT DETECTED:")
print(json.dumps(classification, indent=2))

print("\nDIFFERENCE:")
print("""
Before:
The tutor directly answers the raw student question.

After:
The system first identifies the student's skill,
learning intent, level, and topic.

The generated answer can therefore be more targeted
to the student's actual learning objective.
""")

OPTIMIZATION 2 — BEFORE VS AFTER

BEFORE:
Welcome to the world of drawing. I'm excited to help you improve your sketching skills. As a beginner, it's essential to start with the basics and build your way up. Here are some tips to get you started:

1. **Practice regularly**: Consistency is key when it comes to improving your drawing skills. Set aside a specific time each day or each week to practice drawing, even if it's just for a few minutes.
2. **Start with simple exercises**: Begin with basic exercises like drawing shapes, lines, and values (light and dark). This will help you develop your fine motor skills and get a feel for the medium you're using.
3. **Use reference images**: Gather reference images of the things you want to draw, such as animals, landscapes, or still-life compositions. This will help you observe the proportions, shapes, and details of the subject.
4. **Learn to see and observe**: Drawing is not just about putting lines on paper; it's about observing and interpre

In [ ]:
#optimization 3

In [ ]:
dynamic_prompts = {

    "sketching": """
You are a sketching specialist.

Focus on:
- Basic shapes
- Proportions
- Line control
- Observation
- Sketching exercises
""",

    "shading": """
You are a shading specialist.

Focus on:
- Light source
- Values
- Gradients
- Shadows
- Shading exercises
""",

    "perspective": """
You are a perspective drawing specialist.

Focus on:
- Horizon line
- Vanishing points
- Perspective rules
- Practical exercises
""",

    "general": """
You are an expert drawing tutor.
Give clear and practical drawing guidance.
"""
}

In [ ]:
dynamic_classifier = f"""
Classify this drawing question into exactly one:

sketching
shading
perspective
general

Return ONLY the category.

Question:

{USER_QUERY}
"""

selected_topic = ask_groq(
    dynamic_classifier,
    temperature=0
).strip().lower()

if selected_topic not in dynamic_prompts:
    selected_topic = "general"

print("Selected prompt:", selected_topic)

Selected prompt: sketching


In [ ]:
optimization_3_output = ask_groq(
    dynamic_prompts[selected_topic] +
    "\n\nStudent question:\n" +
    USER_QUERY
)

In [ ]:
print("=" * 100)
print("OPTIMIZATION 3 — BEFORE VS AFTER")
print("=" * 100)

print("\nBEFORE:")
print(baseline_output)

print("\nAFTER:")
print(optimization_3_output)

print("\nSELECTED PROMPT TYPE:")
print(selected_topic)

print("\nDIFFERENCE:")
print("""
Before:
A single generic tutor prompt is used for every
drawing-related question.

After:
The system identifies the drawing task and selects
a specialized prompt.

This allows different prompts to handle sketching,
shading, perspective, and other drawing tasks.
""")

OPTIMIZATION 3 — BEFORE VS AFTER

BEFORE:
Welcome to the world of drawing. I'm excited to help you improve your sketching skills. As a beginner, it's essential to start with the basics and build your way up. Here are some tips to get you started:

1. **Practice regularly**: Consistency is key when it comes to improving your drawing skills. Set aside a specific time each day or each week to practice drawing, even if it's just for a few minutes.
2. **Start with simple exercises**: Begin with basic exercises like drawing shapes, lines, and values (light and dark). This will help you develop your fine motor skills and get a feel for the medium you're using.
3. **Use reference images**: Gather reference images of the things you want to draw, such as animals, landscapes, or still-life compositions. This will help you observe the proportions, shapes, and details of the subject.
4. **Learn to see and observe**: Drawing is not just about putting lines on paper; it's about observing and interpre

In [ ]:
clarification_prompt = f"""
You are a clarification agent for a drawing tutor.

Determine whether the student's question is ambiguous.

If it is ambiguous, ask a useful clarification question.

If it is clear, return:

NO_CLARIFICATION_NEEDED

Student question:

{USER_QUERY}
"""

clarification_result = ask_groq(
    clarification_prompt,
    temperature=0
)

In [ ]:
AMBIGUOUS_QUERY = "Teach me drawing."

In [ ]:
clarification_prompt = f"""
You are a clarification agent for a drawing tutor.

The student said:

{AMBIGUOUS_QUERY}

The request is vague.

Ask one useful clarification question.
For example, determine whether the student means
sketching, shading, perspective, portraits, or another
drawing topic.

Return only the clarification question.
"""

optimization_4_output = ask_groq(
    clarification_prompt,
    temperature=0
)

In [ ]:
print("=" * 100)
print("OPTIMIZATION 4 — CLARIFICATION AGENT")
print("=" * 100)

print("\nBEFORE:")
print("""
Student: Teach me drawing.

Tutor:
[Attempts to answer without knowing what the
student actually wants.]
""")

print("\nAFTER:")
print("""
Student: Teach me drawing.
Tutor:
""" + optimization_4_output)

print("\nDIFFERENCE:")
print("""
Before:
The system attempts to answer an ambiguous question.

After:
The system recognizes ambiguity and asks the student
for clarification before teaching.
""")

OPTIMIZATION 4 — CLARIFICATION AGENT

BEFORE:

Student: Teach me drawing.

Tutor:
[Attempts to answer without knowing what the
student actually wants.]


AFTER:

Student: Teach me drawing.
Tutor:
What specific area of drawing would you like to learn, such as sketching, cartooning, portraits, landscapes, or something else?

DIFFERENCE:

Before:
The system attempts to answer an ambiguous question.

After:
The system recognizes ambiguity and asks the student
for clarification before teaching.



In [ ]:
#opt 5

In [ ]:
initial_response_5 = ask_groq(
    baseline_prompt
)

In [ ]:
validation_prompt = f"""
You are a quality checker for a drawing tutor.

Check whether this response is:

- Relevant
- Clear
- Complete
- Practical
- Well structured

Question:
{USER_QUERY}

Response:
{initial_response_5}

Identify any missing or weak areas.

Return ONLY a short list of improvements.
"""

validation_result = ask_groq(
    validation_prompt,
    temperature=0
)

In [ ]:
optimization_5_prompt = f"""
You are an expert drawing tutor.

Improve the following response based on the quality
check.

Original response:

{initial_response_5}

Quality check:

{validation_result}

Student question:

{USER_QUERY}

Return only the improved tutoring response.
"""

optimization_5_output = ask_groq(
    optimization_5_prompt
)

In [ ]:
print("=" * 100)
print("OPTIMIZATION 5 — RESPONSE VALIDATION")
print("=" * 100)

print("\nBEFORE:")
print(initial_response_5)

print("\nAFTER:")
print(optimization_5_output)

print("\nDIFFERENCE:")
print("""
Before:
The generated answer is returned without checking
whether important information is missing.

After:
The response is checked for relevance, clarity,
completeness, usefulness, and structure.

The detected weaknesses are then used to produce
an improved answer.
""")

OPTIMIZATION 5 — RESPONSE VALIDATION

BEFORE:
Welcome to the world of drawing. As a beginner, it's exciting to start your creative journey. Improving your sketching skills takes practice, patience, and dedication, but don't worry, I'm here to guide you. Here are some tips to help you get started:

1. **Start with the basics**: Understand the fundamentals of drawing, such as lines, shapes, values, and textures. Practice drawing simple shapes like spheres, cylinders, and cubes to get a feel for form and structure.
2. **Practice regularly**: Set aside a specific time each day or each week to practice drawing. Consistency is key to improving your skills. Start with short sessions (15-30 minutes) and gradually increase the time as you become more comfortable.
3. **Use reference images**: Gather reference images from books, online, or real-life observations to help you draw what you see. This will help you develop your observation skills and learn to render different subjects.
4. **Warm up w

In [ ]:
#opt 6

In [ ]:
initial_response_6 = ask_groq(
    baseline_prompt
)

In [ ]:
reflection_prompt = f"""
Review your own drawing tutoring answer.

Student question:
{USER_QUERY}

Your answer:
{initial_response_6}

Think about:

- Missing information
- Incorrect information
- Unclear explanations
- Repetition
- Weak examples
- Poor formatting

List the improvements needed.
"""

In [ ]:
reflection_result = ask_groq(
    reflection_prompt,
    temperature=0
)

In [ ]:
optimization_6_prompt = f"""
You are a drawing tutor improving your own answer.

Original answer:

{initial_response_6}

Self-reflection:

{reflection_result}

Rewrite the answer using the identified improvements.

Student question:

{USER_QUERY}

Return only the improved tutoring answer.
"""

optimization_6_output = ask_groq(
    optimization_6_prompt
)

In [ ]:
print("=" * 100)
print("OPTIMIZATION 6 — SELF REFLECTION")
print("=" * 100)

print("\nBEFORE:")
print(initial_response_6)

print("\nAFTER:")
print(optimization_6_output)

print("\nDIFFERENCE:")
print("""
Before:
The tutor generates an answer and immediately returns it.

After:
The tutor reviews its own answer, identifies weaknesses,
and generates a revised response.
""")

OPTIMIZATION 6 — SELF REFLECTION

BEFORE:
Welcome to the world of drawing. I'm excited to help you improve your sketching skills. As a beginner, it's essential to start with the basics and build your way up. Here are some tips to get you started:

1. **Practice regularly**: Consistency is key when it comes to improving your drawing skills. Set aside a specific time each day or each week to practice drawing.
2. **Start with simple exercises**: Begin with basic exercises like drawing lines, shapes, and values (light and dark). This will help you develop your fine motor skills and get a feel for the drawing tools.
3. **Learn the fundamentals of drawing**: Understand the basics of perspective, proportion, and measurement. This will help you draw more accurately and confidently.
4. **Use reference images**: Look at the work of other artists and use reference images to help you draw. This will give you an idea of how to render different subjects and textures.
5. **Experiment with different t

In [ ]:
#opt 7

In [ ]:
retry_prompt = f"""
You are a drawing tutor.

Answer this question:

{USER_QUERY}

Return a clear and complete response with:

1. Explanation
2. Steps
3. Practical tips
4. Example

Make sure the response is properly formatted.
"""

attempt_1 = ask_groq(
    retry_prompt
)

In [ ]:
retry_check_prompt = f"""
Check whether this drawing tutor response contains:

- Explanation
- Steps
- Practical tips
- Example

Return ONLY:

PASS

or

RETRY
"""

retry_check = ask_groq(
    retry_check_prompt + "\n\nResponse:\n" + attempt_1,
    temperature=0
).strip().upper()

In [ ]:
if "RETRY" in retry_check:

    attempt_2 = ask_groq(
        """
The previous answer was incomplete.

Generate a corrected and complete drawing tutoring
response.

""" + USER_QUERY
    )

    optimization_7_output = attempt_2

else:

    optimization_7_output = attempt_1

In [ ]:
print("=" * 100)
print("OPTIMIZATION 7 — RETRY STRATEGY")
print("=" * 100)

print("\nBEFORE:")
print(baseline_output)

print("\nAFTER:")
print(optimization_7_output)

print("\nDIFFERENCE:")
print("""
Before:
The system uses a single generation attempt.

After:
The system checks the generated response.
If the response fails the required format,
it generates another response.
""")

OPTIMIZATION 7 — RETRY STRATEGY

BEFORE:
Welcome to the world of drawing. I'm excited to help you improve your sketching skills. As a beginner, it's essential to start with the basics and build your way up. Here are some tips to get you started:

1. **Practice regularly**: Consistency is key when it comes to improving your drawing skills. Set aside a specific time each day or each week to practice drawing, even if it's just for a few minutes.
2. **Start with simple exercises**: Begin with basic exercises like drawing shapes, lines, and values (light and dark). This will help you develop your fine motor skills and get a feel for the medium you're using.
3. **Use reference images**: Gather reference images of the things you want to draw, such as animals, landscapes, or still-life compositions. This will help you observe the proportions, shapes, and details of the subject.
4. **Learn to see and observe**: Drawing is not just about putting lines on paper; it's about observing and interpret

In [ ]:
#opt 8

In [ ]:
LEARNING_LEVEL = "beginner"

In [ ]:
adaptive_prompts = {

    "beginner": """
You are teaching a beginner.

Use:
- Simple language
- Basic concepts
- Very clear steps
- Easy exercises
- Encouraging explanations
""",

    "intermediate": """
You are teaching an intermediate learner.

Use:
- Moderate technical detail
- More challenging exercises
- Technique explanations
""",

    "advanced": """
You are teaching an advanced learner.

Use:
- Technical terminology
- Detailed explanations
- Advanced techniques
- Challenging exercises
"""
}

In [ ]:
optimization_8_output = ask_groq(
    adaptive_prompts[LEARNING_LEVEL] +
    "\n\nStudent question:\n" +
    USER_QUERY
)

In [ ]:
print("=" * 100)
print("OPTIMIZATION 8 — ADAPTIVE TUTORING")
print("=" * 100)

print("\nBEFORE:")
print(baseline_output)

print("\nAFTER:")
print(optimization_8_output)

print("\nLEARNING LEVEL:")
print(LEARNING_LEVEL)

print("\nDIFFERENCE:")
print("""
Before:
The same style of explanation may be given to every
student.

After:
The response is adapted to the learner's selected
level.

Beginner explanations use simpler language and easier
exercises, while advanced learners can receive more
technical material.
""")

OPTIMIZATION 8 — ADAPTIVE TUTORING

BEFORE:
Welcome to the world of drawing. I'm excited to help you improve your sketching skills. As a beginner, it's essential to start with the basics and build your way up. Here are some tips to get you started:

1. **Practice regularly**: Consistency is key when it comes to improving your drawing skills. Set aside a specific time each day or each week to practice drawing, even if it's just for a few minutes.
2. **Start with simple exercises**: Begin with basic exercises like drawing shapes, lines, and values (light and dark). This will help you develop your fine motor skills and get a feel for the medium you're using.
3. **Use reference images**: Gather reference images of the things you want to draw, such as animals, landscapes, or still-life compositions. This will help you observe the proportions, shapes, and details of the subject.
4. **Learn to see and observe**: Drawing is not just about putting lines on paper; it's about observing and interp

In [ ]:
#opt 9

In [ ]:
QUESTION_1 = """
How can I improve my shading?
"""

QUESTION_2 = """
How often should I practice it?
"""

In [ ]:
before_context = ask_groq(
    f"""
You are a drawing tutor.

Answer:

{QUESTION_2}
"""
)

In [ ]:
context_prompt = f"""
You are a context-aware drawing tutor.

Previous conversation:

Student:
{QUESTION_1}

Now the student asks:

Student:
{QUESTION_2}

Use the previous conversation to understand what
"it" refers to.

Give a helpful answer.
"""

optimization_9_output = ask_groq(
    context_prompt
)

In [ ]:
print("=" * 100)
print("OPTIMIZATION 9 — CONTEXT AWARENESS")
print("=" * 100)

print("\nPREVIOUS QUESTION:")
print(QUESTION_1)

print("\nFOLLOW-UP QUESTION:")
print(QUESTION_2)

print("\nBEFORE — WITHOUT CONTEXT:")
print(before_context)

print("\nAFTER — WITH CONTEXT:")
print(optimization_9_output)

print("\nDIFFERENCE:")
print("""
Before:
The tutor sees only the latest question and may not
understand what "it" refers to.

After:
The tutor uses the previous conversation to understand
the student's follow-up question.
""")

OPTIMIZATION 9 — CONTEXT AWARENESS

PREVIOUS QUESTION:

How can I improve my shading?


FOLLOW-UP QUESTION:

How often should I practice it?


BEFORE — WITHOUT CONTEXT:
As a drawing tutor, I always recommend that my students practice drawing regularly to see consistent improvement. Ideally, you should aim to practice drawing at least 3-4 times a week, with each session lasting around 30 minutes to an hour.

However, the key is to be consistent and make drawing a habit. Even 15-20 minutes of practice each day can be more beneficial than one or two long sessions a week. This will help you develop your skills and build your confidence as an artist.

Remember, practice doesn't always have to mean creating a complete piece of art. You can practice specific skills, such as drawing shapes, lines, or values, or even just doing quick sketches to warm up your hand and get your creative juices flowing.

The most important thing is to find a schedule that works for you and stick to it. With regula

In [ ]:
#opt 10

In [ ]:
raw_response_10 = ask_groq(
    baseline_prompt
)

In [ ]:
refinement_prompt = f"""
You are a response refinement agent for a drawing tutor.

Improve the following response.

Rules:

- Remove repeated information.
- Simplify complex sentences.
- Improve readability.
- Add useful headings.
- Use bullet points where appropriate.
- Maintain the original meaning.
- Keep important technical information.
- Make the answer beginner-friendly.

Original response:

{raw_response_10}

Return only the refined response.
"""

optimization_10_output = ask_groq(
    refinement_prompt
)

In [ ]:
print("=" * 100)
print("OPTIMIZATION 10 — RESPONSE REFINEMENT")
print("=" * 100)

print("\nBEFORE:")
print(raw_response_10)

print("\nAFTER:")
print(optimization_10_output)

print("\nDIFFERENCE:")
print("""
Before:
The raw generated response may contain repetition,
long sentences, or inconsistent formatting.

After:
The response is cleaned up, simplified, structured,
and made easier to read while preserving its meaning.
""")

OPTIMIZATION 10 — RESPONSE REFINEMENT

BEFORE:
Welcome to the world of drawing. As a beginner, it's great that you're eager to improve your sketching skills. Here are some tips to help you get started:

1. **Practice regularly**: Consistency is key when it comes to improving your drawing skills. Set aside a specific time each day or each week to practice drawing.
2. **Start with simple exercises**: Begin with basic exercises like drawing shapes, lines, and values (different shades of light and dark). This will help you develop your fine motor skills and hand-eye coordination.
3. **Observe and study**: Look at the work of other artists, both professional and amateur. Study the proportions, anatomy, and details of the subjects they draw. Observe the way they use light and shadow, texture, and composition.
4. **Use reference images**: Use reference images to help you draw more accurately. You can find reference images online or use photographs of objects, animals, or people.
5. **Learn th